# LEGO Object Detection Training - Google Colab

**Thesis Project:** DLSU RIAL-3-2425-C7  
**Model:** YOLOv8n for LEGO brick detection  
**GPU Training:** 2-5 hours with Colab GPU (vs 15-30 hours on CPU)

---

## 📋 Before You Start

1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → **T4 GPU** or **A100 GPU**
2. **Upload datasets**: Upload your datasets folder to Google Drive
3. **Run cells in order**: Execute each cell sequentially

---

## 1. Setup Environment

Mount Google Drive and check GPU availability.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted")

In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print("\n✓ GPU is ready! Training will be FAST (2-5 hours)")
else:
    print("\n⚠️ WARNING: No GPU detected! Training will be VERY SLOW (15-30 hours)")
    print("\n** IMPORTANT: Enable GPU now! **")
    print("Go to: Runtime → Change runtime type → Hardware accelerator → GPU (T4 or A100)")
    print("Then restart this cell.")

## 2. Install Dependencies

Install required packages (takes ~2 minutes).

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python pillow pandas matplotlib seaborn pyyaml tqdm scikit-learn xmltodict tensorboard

print("✓ All dependencies installed")

## 3. Setup Project Files

### Method 1: Clone from GitHub (Recommended if code is pushed)

In [ ]:
# OPTION A: Clone from GitHub
# Replace with your actual GitHub repository URL
!git clone https://github.com/daxtangco/block-coding-robot.git
%cd block-coding-robot

print("✓ Project cloned from GitHub")

### Method 2: Upload from Google Drive (Alternative)

In [ ]:
# OPTION B: Copy from Google Drive (uncomment if using this method)
# import os
# from pathlib import Path

# PROJECT_PATH = '/content/drive/MyDrive/block-coding-robot'

# if Path(PROJECT_PATH).exists():
#     %cd $PROJECT_PATH
#     print(f"✓ Project loaded from: {PROJECT_PATH}")
# else:
#     print(f"❌ Project not found at: {PROJECT_PATH}")
#     print("\nPlease upload your project to Google Drive first!")

## 4. Download Datasets from Kaggle

### Step 1: Get Kaggle API Token

1. Go to https://www.kaggle.com/settings/account
2. Scroll to "API" section
3. Click "Create New Token"
4. Download `kaggle.json`
5. Upload `kaggle.json` to Colab (use the file upload button on the left)

In [ ]:
# Upload kaggle.json (run this cell, then click upload and select kaggle.json)
from google.colab import files

print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("\n✓ Kaggle API configured")

### Step 2: Download Datasets

In [ ]:
# Download real-world dataset (spiled-lego-bricks)
!mkdir -p datasets
%cd datasets

print("Downloading real-world LEGO dataset...")
!kaggle datasets download -d igorbezsmertnyi/spiled-lego-bricks
!unzip -q spiled-lego-bricks.zip
!rm spiled-lego-bricks.zip

print("\nDownloading synthetic LEGO dataset...")
!kaggle datasets download -d jeanloouisferey/synthetic-lego-brick-dataset-for-object-detection
!unzip -q synthetic-lego-brick-dataset-for-object-detection.zip
!rm synthetic-lego-brick-dataset-for-object-detection.zip

%cd ..
print("\n✓ Datasets downloaded successfully")

In [ ]:
# Verify dataset structure
from pathlib import Path

datasets_path = Path('datasets')

if datasets_path.exists():
    print("✓ Datasets folder found\n")
    print("Dataset structure:")
    for item in datasets_path.iterdir():
        if item.is_dir():
            count = len(list(item.glob('*')))
            print(f"  - {item.name}: {count} files")
else:
    print("❌ Datasets folder not found!")

## 5. Verify Setup

Check that all required files exist.

In [ ]:
# Verify required files
required_files = [
    'config.py',
    'prepare_datasets.py',
    'train.py',
    'validate.py',
    'test.py'
]

all_found = True
for file in required_files:
    if Path(file).exists():
        print(f"✓ {file}")
    else:
        print(f"❌ {file} not found")
        all_found = False

if all_found:
    print("\n✓ All required files present!")
else:
    print("\n❌ Some files are missing. Please check your repository.")

## 6. Prepare Datasets

Process and validate datasets (5-15 minutes).

In [ ]:
# Prepare datasets for training
!python prepare_datasets.py

## 7. Train Experiment 2 (Recommended)

**Two-stage training:**
- Stage 1: Train on synthetic + real-world data (baseline)
- Stage 2: Fine-tune on real-world data only

**Time with GPU:** 2-4 hours  
**Time with CPU:** 15-30 hours

### Important: This will run for hours - keep the browser tab open!

In [ ]:
# Train Experiment 2 (with live progress)
!python train.py --experiment 2

## 8. Validate Model

Check model performance on test set.

In [ ]:
# Find the best model from Experiment 2
import glob
from pathlib import Path

# Try Stage 2 first (fine-tuned), fallback to Stage 1
exp2_models = glob.glob('training_output/models/experiment_2_*/stage2_finetuned/weights/best.pt')
if not exp2_models:
    print("Stage 2 model not found, checking Stage 1...")
    exp2_models = glob.glob('training_output/models/experiment_2_*/stage1_synthetic/weights/best.pt')

if exp2_models:
    exp2_model = exp2_models[-1]  # Get most recent
    print(f"Found model: {exp2_model}")
    !python validate.py --model $exp2_model --experiment 2
else:
    print("❌ No trained model found. Run training first.")

## 9. View Training Results

Visualize training curves and validation metrics.

In [ ]:
from IPython.display import Image, display
import json

print("=" * 70)
print("TRAINING RESULTS")
print("=" * 70)

if exp2_models:
    exp2_dir = Path(exp2_models[-1]).parent.parent
    
    # Show training curve
    results_plot = exp2_dir / 'results.png'
    if results_plot.exists():
        print("\nTraining Curves:")
        display(Image(filename=str(results_plot), width=800))
    
    # Show confusion matrix
    confusion_matrix = exp2_dir / 'confusion_matrix_normalized.png'
    if confusion_matrix.exists():
        print("\nConfusion Matrix:")
        display(Image(filename=str(confusion_matrix), width=600))

# Show validation metrics
print("\n" + "=" * 70)
print("VALIDATION METRICS")
print("=" * 70)

metrics_files = glob.glob('training_output/results/*/metrics.json')
if metrics_files:
    with open(metrics_files[-1], 'r') as f:
        metrics = json.load(f)
    
    print(f"\nmAP@0.5:       {metrics['mAP@0.5']:.4f}")
    print(f"mAP@0.5:0.95:  {metrics['mAP@0.5:0.95']:.4f}")
    print(f"Precision:     {metrics['precision']:.4f}")
    print(f"Recall:        {metrics['recall']:.4f}")
    print(f"F1 Score:      {metrics['F1']:.4f}")
    
    # Check success criteria
    print("\n" + "=" * 70)
    print("SUCCESS CRITERIA CHECK")
    print("=" * 70)
    
    criteria = [
        ('mAP@0.5', metrics['mAP@0.5'], 0.70),
        ('Precision', metrics['precision'], 0.75),
        ('Recall', metrics['recall'], 0.70)
    ]
    
    all_passed = True
    for name, value, target in criteria:
        status = "✓ PASS" if value >= target else "✗ FAIL"
        print(f"{name:12s}: {value:.4f} / {target:.2f} [{status}]")
        if value < target:
            all_passed = False
    
    print(f"\nOverall: {'✓ ALL CRITERIA MET!' if all_passed else '✗ Some criteria not met'}")
    
    # Show per-class performance
    print("\nPer-Class AP@0.5:")
    for cls, ap in metrics['per_class_AP'].items():
        print(f"  {cls:12s}: {ap:.4f}")
    
    # Show per-class plot
    per_class_plot = Path(metrics_files[-1]).parent / 'per_class_performance.png'
    if per_class_plot.exists():
        print("\nPer-Class Performance:")
        display(Image(filename=str(per_class_plot), width=700))

## 10. Test Model on Images

Run inference on test images to see predictions.

In [ ]:
# Test on a single image
if exp2_models:
    exp2_model = exp2_models[-1]
    
    # Find a test image
    test_images = list(Path('datasets/images').glob('*.jpg'))[:5]
    
    if test_images:
        test_image = str(test_images[0])
        print(f"Testing on: {test_image}\n")
        !python test.py --model $exp2_model --image $test_image --visualize --benchmark
    else:
        print("No test images found")
else:
    print("No trained model found.")

## 11. Export Model

Export to TFLite for ESP32-CAM deployment.

In [ ]:
# Export best model to TFLite
if exp2_models:
    exp2_model = exp2_models[-1]
    !python export_model.py --model $exp2_model --format tflite
    print("\n✓ Model exported to TFLite format for ESP32-CAM")
else:
    print("No trained model found.")

## 12. Save Results to Google Drive

**IMPORTANT:** Results are automatically saved to your Google Drive at:
`/content/drive/MyDrive/block-coding-robot/training_output/`

You can also create a backup zip file:

In [ ]:
# Create backup zip of results
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f'lego_detection_results_{timestamp}'

if Path('training_output').exists():
    print("Creating backup zip file...")
    shutil.make_archive(zip_name, 'zip', 'training_output')
    
    zip_size = Path(f'{zip_name}.zip').stat().st_size / (1024*1024)
    print(f"\n✓ Backup created: {zip_name}.zip")
    print(f"  Size: {zip_size:.1f} MB")
    
    # Copy to Google Drive
    drive_backup_dir = Path('/content/drive/MyDrive/LEGO_Training_Backups')
    drive_backup_dir.mkdir(exist_ok=True)
    
    shutil.copy(f'{zip_name}.zip', str(drive_backup_dir))
    print(f"\n✓ Backup also saved to Google Drive:")
    print(f"  {drive_backup_dir}/{zip_name}.zip")
else:
    print("No training output found.")

## 📊 Training Complete!

### Your Results:

✅ **Trained model saved to:**
- Google Drive: `/content/drive/MyDrive/block-coding-robot/training_output/models/`
- Best model: `experiment_2_*/stage2_finetuned/weights/best.pt`

✅ **Files for thesis:**
- `results.csv` - Training metrics over time
- `metrics.json` - Final validation results
- `results.png` - Training curves
- `confusion_matrix.png` - Model confusion matrix
- `per_class_performance.png` - Per-class AP scores

✅ **Exported models:**
- `best.pt` - PyTorch model
- `best.tflite` - TFLite for ESP32-CAM

### Next Steps:

1. ✅ Download your trained model from Google Drive
2. ✅ Test on real LEGO bricks with your camera
3. ✅ Integrate with ESP32-CAM and robotic arm
4. ✅ Document results in thesis

---

**Good luck with your project! 🎓🤖**

**Questions?** Check the documentation or contact your groupmates.